# Module 2 — SOLUTION: Vector Store & Retrieval

> **Instructor note:** This is the complete solution. Share after the exercise session.

In [ ]:
import json
import time
from pathlib import Path

import chromadb
import matplotlib.pyplot as plt
import numpy as np
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

with open("../data/raw/corpus.json") as f:
    corpus = json.load(f)

def chunk_fixed_size(text: str, chunk_size: int = 200, overlap: int = 20) -> list[str]:
    words = text.split()
    step = chunk_size - overlap
    return [" ".join(words[i : i + chunk_size]) for i in range(0, len(words), step) if words[i:i+chunk_size]]

all_chunks = []
for article in corpus:
    for i, text in enumerate(chunk_fixed_size(article["content"])):
        all_chunks.append({"text": text, "source": article["title"], "id": f"{article['title']}_{i}"})

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
print(f"Ready: {len(all_chunks)} chunks, embedding model loaded")

In [ ]:
# Exercise 2.1 solution — Create persistent ChromaDB collection
CHROMA_PATH = "../chroma_db"

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = chroma_client.get_or_create_collection(
    name="workshop_rag",
    metadata={"hnsw:space": "cosine"},
)
print(f"Collection '{collection.name}' has {collection.count()} documents")

In [ ]:
# Exercise 2.2 solution — Batch indexing
BATCH_SIZE = 500

def index_chunks(collection, chunks: list[dict], embed_model, batch_size: int = BATCH_SIZE) -> None:
    # Deduplicate by id (dataset may contain duplicate documents)
    seen = {}
    for c in chunks:
        seen[c["id"]] = c
    unique_chunks = list(seen.values())
    if len(unique_chunks) < len(chunks):
        print(f"Deduplicated {len(chunks) - len(unique_chunks)} duplicate chunks")

    print(f"Indexing {len(unique_chunks)} chunks in batches of {batch_size}...")
    start = time.time()

    for i in tqdm(range(0, len(unique_chunks), batch_size)):
        batch = unique_chunks[i : i + batch_size]
        texts = [c["text"] for c in batch]
        ids = [c["id"] for c in batch]
        metadatas = [{"source": c["source"]} for c in batch]
        embeddings = embed_model.encode(texts, show_progress_bar=False).tolist()

        collection.upsert(
            documents=texts,
            embeddings=embeddings,
            ids=ids,
            metadatas=metadatas,
        )

    elapsed = time.time() - start
    print(f"Indexed {len(unique_chunks)} chunks in {elapsed:.1f}s")


index_chunks(collection, all_chunks, embed_model)
print(f"Collection now has {collection.count()} documents")

In [ ]:
# Exercise 2.3 solution — Retrieval function
def retrieve(question: str, top_k: int = 5) -> list[dict]:
    q_embedding = embed_model.encode(question).tolist()

    results = collection.query(
        query_embeddings=[q_embedding],
        n_results=top_k,
        include=["documents", "metadatas", "distances"],
    )

    retrieved = []
    for text, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    ):
        retrieved.append({
            "text": text,
            "source": meta["source"],
            "similarity": 1 - dist,  # cosine distance → similarity
        })

    return retrieved


question = "What hospitalization costs does DKV cover?"
results = retrieve(question, top_k=5)
print(f"Query: {question}\n")
for i, r in enumerate(results, 1):
    print(f"[{i}] {r['source']} | sim={r['similarity']:.3f}")
    print(f"    {r['text'][:300]}...\n")

In [ ]:
# Exercise 2.4 solution — Threshold filtering
def retrieve_filtered(question: str, top_k: int = 5, min_similarity: float = 0.3) -> list[dict]:
    results = retrieve(question, top_k=top_k)
    return [r for r in results if r["similarity"] >= min_similarity]


# Visualise similarity scores
test_questions = [
    "What hospitalization costs does DKV cover?",
    "What dental treatments does DKV Smile cover?",
    "What does DKV Home Care insurance provide?",
    "What is the capital of France?",
]

fig, axes = plt.subplots(1, len(test_questions), figsize=(16, 4))
for ax, question in zip(axes, test_questions):
    results = retrieve(question, top_k=10)
    similarities = [r["similarity"] for r in results]
    sources = [r["source"][:20] for r in results]
    ax.barh(range(len(similarities)), similarities, color="steelblue")
    ax.set_yticks(range(len(sources)))
    ax.set_yticklabels(sources, fontsize=7)
    ax.set_xlim(0, 1)
    ax.axvline(0.4, color="red", linestyle="--", alpha=0.5)
    ax.set_title(question[:40] + "...", fontsize=8)
    ax.set_xlabel("Similarity")
plt.tight_layout()
plt.show()

In [ ]:
# Exercise 2.5 solution — Metadata filtering
def retrieve_from_source(question: str, source: str, top_k: int = 5) -> list[dict]:
    q_embedding = embed_model.encode(question).tolist()

    results = collection.query(
        query_embeddings=[q_embedding],
        n_results=top_k,
        where={"source": source},
        include=["documents", "metadatas", "distances"],
    )

    return [
        {"text": text, "source": meta["source"], "similarity": 1 - dist}
        for text, meta, dist in zip(
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0],
        )
    ]


results = retrieve_from_source("What accommodation costs are covered during hospitalisation?", source="dkv_hospitalisation_en", top_k=3)
print(f"Results from 'dkv_hospitalisation_en' only: {len(results)}")
for r in results:
    print(f"  sim={r['similarity']:.3f} | {r['text'][:100]}...")